# Figure 5 — touch option pricing

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fast_excursion_limit import plot_config
from fast_excursion_limit.fast_excursion_heston import FastExcursionHeston
from fast_excursion_limit.selections import selection_process

plot_config.set_style()

SAVE_PLOT = True

## Parameters

In [ ]:
from fast_excursion_limit import defaults

seed = defaults.SEED

maturity = defaults.MATURITY_TOUCHES
delta_scale = defaults.DELTA_SCALE_TOUCHES
num_paths = defaults.NUM_PATHS_TOUCHES

num_strikes = 1000
strike_delta = 0.05  # forward delta bounding the strike grid width, both sides

num_digital_maturities = 10

spot_price = 1.16
domestic_rate = 0.05
foreign_rate = 0.02
sigma = 0.07
rho = 0.05
gamma = 0.20
filename = "figure-5.pdf"

## Simulate paths

In [ ]:
step_size = delta_scale * maturity

model = FastExcursionHeston(
    spot_price=spot_price,
    domestic_rate=domestic_rate,
    foreign_rate=foreign_rate,
    sigma=sigma,
    rho=rho,
    gamma=gamma,
)

np.random.seed(seed)
feh_lows = np.empty(num_paths)
feh_highs = np.empty(num_paths)
frh_lows = np.empty(num_paths)
frh_highs = np.empty(num_paths)
terminals = np.empty(num_paths)
for path in range(num_paths):
    Z, Y = model.simulate(maturity=maturity, step_size=step_size)
    feh_lows[path] = Z.min()
    feh_highs[path] = Z.max()

    Z_close = selection_process(Z, Y, "close")
    frh_lows[path] = Z_close.min()
    frh_highs[path] = Z_close.max()
    terminals[path] = np.interp(maturity, Y, Z_close)

## Price single-touch, no-touch, and digital put options

In [ ]:
probabilities = np.linspace(strike_delta, 1 - strike_delta, num_strikes)
strikes = model.ppf(probabilities, maturity)
strikes_down = strikes[strikes <= model.spot_price]
strikes_up = strikes[strikes >= model.spot_price]

feh_single_touch = (feh_lows[:, None] <= strikes_down[None, :]).mean(axis=0)
feh_no_touch = 1 - (feh_highs[:, None] >= strikes_up[None, :]).mean(axis=0)
frh_single_touch = (frh_lows[:, None] <= strikes_down[None, :]).mean(axis=0)
frh_no_touch = 1 - (frh_highs[:, None] >= strikes_up[None, :]).mean(axis=0)

digital_maturities = (
    np.linspace(1 / num_digital_maturities, 1, num_digital_maturities) * maturity
)
digitals = [model.digital_put_forward_price(strikes, t) for t in digital_maturities]

simulated_final_digital = (terminals[:, None] <= strikes[None, :]).mean(axis=0)

diff_down = feh_single_touch - frh_single_touch
diff_up = feh_no_touch - frh_no_touch
i_down, i_up = np.argmax(np.abs(diff_down)), np.argmax(np.abs(diff_up))
max_diff_down = strikes_down[i_down], feh_single_touch[i_down], frh_single_touch[i_down]
max_diff_up = strikes_up[i_up], feh_no_touch[i_up], frh_no_touch[i_up]

## Plot

In [ ]:
fig, ax = plt.subplots(figsize=plot_config.TWO_PANEL_FIGSIZE)

for digital in digitals[:-1]:
    ax.plot(strikes, digital, color="gray", alpha=0.3, lw=0.5)
(digital_analytic,) = ax.plot(
    strikes, digitals[-1], color="gray", alpha=0.3, lw=0.5, label="Digital analytic"
)
(digital_simulated,) = ax.plot(
    strikes,
    simulated_final_digital,
    color="gray",
    alpha=0.6,
    ls=":",
    lw=1,
    label="Digital simulated",
)

(feh_st,) = ax.plot(strikes_down, feh_single_touch, label="FEH single-touch")
(feh_nt,) = ax.plot(strikes_up, feh_no_touch, label="FEH no-touch")
(frh_st,) = ax.plot(strikes_down, frh_single_touch, label="FRH single-touch")
(frh_nt,) = ax.plot(strikes_up, frh_no_touch, label="FRH no-touch")

cap = 0.004 * (strikes[-1] - strikes[0])
for strike, y_feh, y_frh in (max_diff_down, max_diff_up):
    ax.plot([strike - cap, strike + cap], [y_feh, y_feh], color="gray", lw=0.8)
    ax.plot([strike - cap, strike + cap], [y_frh, y_frh], color="gray", lw=0.8)
    ax.plot([strike, strike], [y_frh, y_feh], color="gray", lw=0.8)
    ax.annotate(
        f"{abs(y_feh - y_frh) * 100:.0f}%",
        xy=(strike, (y_feh + y_frh) / 2),
        xytext=(3, 0),
        textcoords="offset points",
        fontsize=7,
        color="gray",
        va="center",
    )

ax.set_xlabel("Barrier")
ax.set_ylabel("Touch option price")

touch_legend = ax.legend(
    handles=[feh_st, frh_st, frh_nt, feh_nt], loc="upper left", fontsize=7
)
ax.add_artist(touch_legend)
ax.legend(handles=[digital_analytic, digital_simulated], loc="lower right", fontsize=7)

fig.tight_layout()
if SAVE_PLOT:
    fig.savefig(plot_config.PLOTS_DIR / filename)